# README
- **Author**: RUEI-HSIANG, CHANG（B12303030；AI協助）
- **Created At**: 2026-10-19
- **Last Modified At**: 2026-10-19

## What does this file do?
第八輪跨公職、跨屆交叉核對：黃裕庭以當屆自述仁政里第3、4屆里長，連到2022官方里長名冊，行政區代碼逐層對應高雄苓雅仁政里；陳傑麟以當屆明載宜蘭第18、19屆議員連接2014、2018及2022官方山原議員紀錄；陳泓維以當屆明載2020桃園第一選區立委參選人，連接已核對2020與2022一致生日／性別。非單憑同名。歷史學歷為申報級別，不認證2026最高學位。

## What does this file take?
固定4份第七輪快照及PINS列示的CEC CSV、行政區碼與當屆競選頁原文。來源文件：[原始說明](../raw/cross_office_round8_20261019_documentation.ipynb)。不執行網路請求。

## What does this file output?
4份current、3人逐項審閱表與逐欄證據，共6個CSV。原始資料不變。資料角色review_inventory/evidence，非最終分析樣本。

## Known limitations
里長參選不能當作曾參選議員；2022參選／當選不證明2026仍在任。社群帳號及內容本輪不變，Meta與貼文窗口暫緩。人物網頁缺口仍730人。


第九輪接續：本輪3份current整合表封存至pre_municipal_round9快照；舊程式不覆寫最新背景表。待核對歷史工作單本輪未改，仍151人。

In [1]:
from pathlib import Path
import json,hashlib,csv,datetime,re,unicodedata
import pandas as pd
from bs4 import BeautifulSoup
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'AGENTS.md').exists() and (p/'data').exists())
sha=lambda p:hashlib.sha256((ROOT/p).read_bytes()).hexdigest()
def read(p):return pd.read_csv(ROOT/p,dtype=str,keep_default_na=False)
def norm(s):return re.sub(r'\s+','',unicodedata.normalize('NFKC',s))
PINS={'data/processed/snapshots/pre_cross_office_round8_20261019/candidate_master_current.csv': '0d2d2620f62cde3abaf691ce843925231bc84f95362562b16794d74aadf022e9', 'data/processed/snapshots/pre_cross_office_round8_20261019/candidate_fields_current.csv': 'e2e4528410ee10272ddef071f431e42334e14891255277a1a44dce2203ef9b22', 'data/processed/snapshots/pre_cross_office_round8_20261019/council_background_20261019.csv': 'f4b9055e53c7d8476f34389e4d810e26569c394c5e4916c50230b31fef7db2da', 'data/processed/snapshots/pre_cross_office_round8_20261019/council_history_pending_identity_current.csv': 'abf997cfa9ceac55cd7636db7950cfa934cb5d553183f5ff05d6181bc9bd72e0', 'data/raw/cec_all_offices/107_elcand.csv': 'af38d1e0ca691d9e353fcfb3e154f0cbe1f178d6e796ce7441ef7cb52d02f1b9', 'data/raw/cec_all_offices/110_elcand.csv': 'b323d6713df428b57161e3b1b6799ebf25d679ef9948e8fe4fbb3fa0d7cfbc73', 'data/raw/cec_all_offices/112_elcand.csv': 'f355705254d5ca457d4fe130b77983b769602126b10713e7c968889d2d52530c', 'data/raw/cec_all_offices/095_elcand.csv': '2106bc9400f7d64784595799d2404496186cffaa56d82b14554d172d781f1b60', 'data/raw/cec_all_offices/073_elcand.csv': '3d5391d0276093fb4721778d84b8d36c4569457d1d865ee14d45805939d9e5bd', 'data/raw/cec_all_offices/087_elcand.csv': '58616de64001f72a3f16be830a15aceb42cecdc63b239ce234c13c3ba408e307', 'data/raw/cec_all_offices_manifest.json': '726d464020ff8f850ce39685bb70926c8529712f75c1833ac5b5c6d795354ed7', 'data/raw/cross_office_round8_20261019/2022_village_elbase.csv': '4d0afbae084ccd77785ebe9a878eadb7e97e828555926d92866dddbfb4b14e79', 'data/raw/cross_office_round8_20261019/manifest.json': '6c3b1612b234da8a0ab7e799fe7367bc6ebaae9ef48f190144fbb09ed4124022', 'data/raw/social_followup_round6_20261019/manifest.json': 'f5471133039402a9e21143f23d30b70b645ddf3406805e19f78764f5b2063746', 'data/raw/kmtks_web_20261019/manifest.json': 'a519c5d80a86591bec6da77b7f031e56c5c10fa9e79245e5e1b74332a4709fb8', 'data/raw/social_followup_round6_20261019/b39ba23e64e7bb9ca8b3.html': 'c0a7c18159f0dc6e66dfcca6757183d0eb26f66487871cd5fb635c1c4e72bdd6', 'data/raw/social_followup_round6_20261019/0277960ed4f266884bad.html': 'f31d0165ee321abb26ce8b9cb6816b9dbbd4655fc9a7f9ddc309a75bcbc12c90', 'data/raw/kmtks_web_20261019/05.txt': '410ff727927bf36c12bc505782951ead7cfb91b88b0e1d28e4802249c7f99b26'}
S='data/processed/snapshots/pre_cross_office_round8_20261019'
for p,h in PINS.items():assert sha(p)==h,p
master=read(S+'/candidate_master_current.csv');fields=read(S+'/candidate_fields_current.csv');council=read(S+'/council_background_20261019.csv');pending=read(S+'/council_history_pending_identity_current.csv')
base=council.copy(deep=True)
assert len(council)==1502 and len(pending)==153
source_records=[]
for f in ['data/raw/social_followup_round6_20261019/manifest.json','data/raw/kmtks_web_20261019/manifest.json']:source_records+=json.loads((ROOT/f).read_text())
def page(url):
 a=next(x for x in source_records if x['url']==url);assert a['path'] in PINS
 return BeautifulSoup((ROOT/a['path']).read_text(),'html.parser').get_text(' ',strip=True),a['path']
def records(path):
 with (ROOT/path).open(encoding='utf-8-sig') as f:return [[x.strip().lstrip("'") for x in row] for row in csv.reader(f)]
def one(path,name):
 hits=[(i+1,r) for i,r in enumerate(records(path)) if len(r)>9 and r[6]==name];assert len(hits)==1,(path,name);return hits[0]
def dob(row):
 s=row[9].zfill(7);return datetime.date(int(s[:3])+1911,int(s[3:5]),int(s[5:7])).isoformat()
def setfield(key,f,val,status,src,note):
 m=fields.candidate_key.eq(key)&fields.field.eq(f);assert m.sum()==1
 fields.loc[m,['existing_value','status','evidence','note']]=[val,status,src,note];master.loc[master.candidate_key.eq(key),f+'_status']=status
obs=[];review=[]
def ev(key,name,f,val,path,row,url,note):obs.append({'candidate_key':key,'candidate_name':name,'field':f,'value':val,'source_file':path,'source_row':str(row),'source_url':url,'evidence_status':note,'checked_date':'2026-10-19'})
print('Fixed source and snapshot hashes verified:',len(PINS))


Fixed source and snapshot hashes verified: 18


In [2]:
# Current village-head biography -> exact CEC geographic codes -> official birthday.
name='黃裕庭';m=council.candidate_name.eq(name)&council.county_city.eq('高雄市');assert m.sum()==1;c=council[m].iloc[0];key=c.candidate_key
url='https://2026.kmtks.org.tw/Council/huang-yu-ting';body,profile=page(url)
assert norm('第3、4 屆仁政里里長') in norm(body)
p='data/raw/cec_all_offices/112_elcand.csv';line,r=one(p,name);assert r[:5]==['64','000','00','080','0014'] and r[8]=='1' and r[12]=='碩士'
geo=records('data/raw/cross_office_round8_20261019/2022_village_elbase.csv')
assert ['64','000','00','000','0000','高雄市'] in geo
assert ['64','000','00','080','0000','苓雅區'] in geo
assert ['64','000','00','080','0014','仁政里'] in geo
birth=dob(r);assert birth=='1981-01-01'
assert c.birth_date_additional_official=='' and c.sex_additional_source_value=='男'
council.loc[m,['birth_date_additional_official','birth_date_additional_source_url','birth_date_additional_source_year','birth_date_additional_evidence_status','sex_additional_source_value','sex_additional_source_url','sex_additional_source_year','additional_historical_education_cec']]=[birth,'https://db.cec.gov.tw/','2022','2022 official village-head registration; current explicit prior Ren-Zheng village-head role and exact geographic code identity support','男','https://db.cec.gov.tw/','2022','碩士']
setfield(key,'birth_date',birth,'2022官方里長申報完整生日，當屆職務與行政區碼交叉支持','https://db.cec.gov.tw/ | '+url,'高雄市苓雅區仁政里64/000/00/080/0014；不是議員參選紀錄；1981-01-01照官方原文不推測時間精度')
setfield(key,'sex','男','2022官方里長申報明載性別，當屆職務交叉支持','https://db.cec.gov.tw/ | '+url,'原競選頁自述男保留在第七輪證據，新增政府歷史申報支持')
for f,val in [('birth_date',birth),('sex','男'),('education','碩士')]:ev(key,name,f,val,p,line,'https://db.cec.gov.tw/','2022里長名冊；對應當屆仁政里里長自述；非2026戶籍／學位認證')
review.append({'candidate_key':key,'candidate_name':name,'decision':'supported_cross_office_village_history','birth_date':birth,'sex':'男','history_file':p,'history_row':str(line),'identity_url':url,'identity_file':profile,'identity_anchor':'高雄苓雅仁政里第3、4屆里長，官方行政區碼逐層一致','first_time_council_conclusion':'unknown; village election does not establish prior council candidacy'})
# Existing 2020 official-linked birthday independently aligns with 2022 council row.
name='陳泓維';m=council.candidate_name.eq(name)&council.county_city.eq('桃園市');c=council[m].iloc[0];key=c.candidate_key
url='https://youth.pfpnext.com/2026/candidate-detail.php?id=pfp2026-003';body,profile=page(url);assert '2020 桃園市第一選區立委參選人' in body
p0='data/raw/cec_all_offices/095_elcand.csv';l0,r0=one(p0,name);assert r0[:3]==['68','000','01'];assert dob(r0)==c.birth_date_additional_official=='1987-05-08'
h=pending[pending.candidate_key.eq(key)].iloc[0];p=h.history_file;line,r=one(p,name);assert str(line)==h.history_row and r[:3]==['68','000','02'] and r[8]=='1'
assert dob(r)==dob(r0)==h.birth_date_history_candidate;assert r[12]==r0[12]=='高中(職)'
# Current explicit 18th/19th council terms -> 2014/2018 mountain-indigenous council records -> 2022 same full DOB.
bridges=[{'name':name,'key':key,'mask':m,'row':r,'line':line,'path':p,'birth':dob(r),'url':url,'profile':profile,'anchor':'當屆明載2020桃園第1選區立委參選；2020與2022生日、性別及學歷級別一致','prior':p0}]
name='陳傑麟';m=council.candidate_name.eq(name)&council.county_city.eq('宜蘭縣');c=council[m].iloc[0];key=c.candidate_key
url='https://youth.pfpnext.com/2026/candidate-detail.php?id=pfp2026-005';body,profile=page(url);assert '第18屆、第19屆縣議員' in body
prior=[]
for p0 in ['data/raw/cec_all_offices/073_elcand.csv','data/raw/cec_all_offices/087_elcand.csv']:
 l0,r0=one(p0,name);assert r0[:3]==['10','002','12'] and r0[8]=='1' and r0[14]=='*' and dob(r0)=='1965-11-08';prior.append(p0)
h=pending[pending.candidate_key.eq(key)].iloc[0];p=h.history_file;line,r=one(p,name);assert str(line)==h.history_row and r[:3]==['10','002','12'] and dob(r)=='1965-11-08' and r[8]=='1' and r[12]=='碩士'
bridges.append({'name':name,'key':key,'mask':m,'row':r,'line':line,'path':p,'birth':dob(r),'url':url,'profile':profile,'anchor':'當屆明載宜蘭第18、19屆議員；2014、2018同選區當選紀錄及2022完整生日／性別一致','prior':' | '.join(prior)})
for b in bridges:
 m=b['mask'];c=council[m].iloc[0];key=b['key'];r=b['row'];birth=b['birth'];assert c.birth_date_supported_history==''
 council.loc[m,['birth_date_supported_history','birth_year_supported_history','sex_supported_history','history_source_file','history_source_row','history_identity_review','birth_sex_evidence_status','education_2022_declared','first_time_this_office_supported','first_time_any_office_supported']]=[birth,birth[:4],'男',b['path'],str(b['line']),b['anchor'],'2022 official declaration cross-year identity corroboration; not 2026 household verification',r[12],'0','0']
 if not c.education_raw:
  council.loc[m,'education_raw']=r[12];council.loc[m,'background_source_url']='https://db.cec.gov.tw/'
  setfield(key,'education',r[12],'2022官方參選申報學歷級別已有身分支持；非2026最高學位認證','https://db.cec.gov.tw/ | '+b['url'],b['anchor']+'；僅保存官方學歷級別，不虛構學校或畢業狀態')
 for f,val in [('birth_date',birth),('sex','男'),('education',r[12]),('first_time_this_office','0'),('first_time_any_office','0')]:
  ev(key,b['name'],f,val,b['path'],b['line'],'https://db.cec.gov.tw/',b['anchor'])
  if f!='education':setfield(key,f,val,'2022官方參選紀錄與當屆履歷跨年支持','https://db.cec.gov.tw/ | '+b['url'],b['anchor'])
 review.append({'candidate_key':key,'candidate_name':b['name'],'decision':'supported_2022_history_cross_year_bridge','birth_date':birth,'sex':'男','history_file':b['path'],'history_row':str(b['line']),'identity_url':b['url'],'identity_file':b['profile'],'identity_anchor':b['anchor'],'prior_official_files':b['prior'],'first_time_council_conclusion':'nonfirst based on 2022 council registration'})
pending=pending[~pending.candidate_key.isin([b['key'] for b in bridges])].copy();assert len(pending)==151
review=pd.DataFrame(review).fillna('');evidence=pd.DataFrame(obs)
print(review[['candidate_name','birth_date','decision']].to_string(index=False))


candidate_name birth_date                                 decision
           黃裕庭 1981-01-01   supported_cross_office_village_history
           陳泓維 1987-05-08 supported_2022_history_cross_year_bridge
           陳傑麟 1965-11-08 supported_2022_history_cross_year_bridge


In [3]:
assert council[['candidate_key','party_registration_raw','incumbency_2026']].equals(base[['candidate_key','party_registration_raw','incumbency_2026']])
assert council.birth_date_supported_history.ne('').sum()==703
assert (council.birth_date_supported_history.ne('')|council.birth_date_additional_official.ne('')).sum()==707
assert (council.sex_supported_history.ne('')|council.sex_additional_source_value.ne('')).sum()==707
assert (council.moi_profile_url.eq('')&council.campaign_profile_url.eq('')&council.additional_profile_url.eq('')).sum()==730
assert council.first_time_any_office_supported.eq('0').sum()==712
assert council.first_time_this_office_supported.eq('0').sum()==705
outputs={'data/processed/snapshots/pre_municipal_round9_20261019/candidate_master_current.csv':master,'data/processed/snapshots/pre_municipal_round9_20261019/candidate_fields_current.csv':fields,'data/processed/snapshots/pre_municipal_round9_20261019/council_background_20261019.csv':council,'data/processed/council_history_pending_identity_current.csv':pending,'data/processed/council_identity_review_round8_20261019.csv':review,'data/processed/candidate_evidence_round8_20261019.csv':evidence}
checks={}
for p,d in outputs.items():
 assert p not in PINS
 d=(d.map if hasattr(d,'map') else d.applymap)(lambda x:str(x).replace('\r','').replace('\n','\\n'))
 d.to_csv(ROOT/p,index=False,encoding='utf-8-sig');saved=read(p);assert saved.shape==d.shape and list(saved.columns)==list(d.columns)
 key=['candidate_key','field'] if p.endswith('candidate_fields_current.csv') else [] if p.endswith('candidate_evidence_round8_20261019.csv') else ['candidate_key']
 if key:assert not saved.duplicated(key).any()
 checks[p]={'rows':len(saved),'columns':len(saved.columns),'key':key,'sha256':sha(p)}
summary={'new_complete_official_birthdays':['黃裕庭','陳傑麟'],'new_2022_council_identity_links':['陳泓維','陳傑麟'],'historical_2022_birth_dates':703,'birth_official_union':707,'sex_explicit_source_union':707,'remaining_profile_gaps':730,'pending_identity':151,'nonfirst_any':712,'nonfirst_council':705,'outputs':checks}
(ROOT/'data/code/cross_office_round8_20261019_verification.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2));print(json.dumps(summary,ensure_ascii=False,indent=2))


{
  "new_complete_official_birthdays": [
    "黃裕庭",
    "陳傑麟"
  ],
  "new_2022_council_identity_links": [
    "陳泓維",
    "陳傑麟"
  ],
  "historical_2022_birth_dates": 703,
  "birth_official_union": 707,
  "sex_explicit_source_union": 707,
  "remaining_profile_gaps": 730,
  "pending_identity": 151,
  "nonfirst_any": 712,
  "nonfirst_council": 705,
  "outputs": {
    "data/processed/snapshots/pre_municipal_round9_20261019/candidate_master_current.csv": {
      "rows": 1583,
      "columns": 34,
      "key": [
        "candidate_key"
      ],
      "sha256": "4fba9288b89aa417734ea686b09ccad5eed147b47162265c799319bd025fa926"
    },
    "data/processed/snapshots/pre_municipal_round9_20261019/candidate_fields_current.csv": {
      "rows": 31660,
      "columns": 11,
      "key": [
        "candidate_key",
        "field"
      ],
      "sha256": "80d9ccb7ea2cb1e5adfd3488f0b3d16581b067311d9e230f3daee37cf7b059ad"
    },
    "data/processed/snapshots/pre_municipal_round9_20261019/council_backgrou